## Caching the Embeddings for each Presentation Slide in our Training Material 
For each Zenodo Record in our Resources "nfdi4bioimage.yml" File, we can generate different embeddings (visual, text and mixed). 
Instead of Calculating it over and over again for different tasks, we can calculate the Embeddings once and store them somewhere (e.g. via Huggingface) to load them again at any time.
For now, the Embeddings are stored as a Dataset on Huggingface.

To get started, you have to choose between using the free Github Models, your own API key (from OpenAI) or the SCADS.AI API key. For that, adjust the variable __use_api__ in the cell below. If neccessary, also edit the name of the API Key (as stored in your environment).

In [ ]:
import sys
import os

# Add the root directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
from huggingface_hub import login
import os

#Authenticate your current session
login(token=os.getenv("HF_TOKEN"))

In [ ]:
# Decide on whether to use GH Models or OpenAI API
use_api = "use_scads_api"  # alternatives: use_gh_models, use_openai

if use_api == "use_gh_models":
    token = os.environ["GITHUB_TOKEN"]
    
if use_api == "use_openai":
    token = os.environ["OPENAI_API_KEY"]

if use_api == "use_scads_api":
    token = os.environ["SCADSAI_API_KEY"]

1. Create lists of valid licenses to filter out unwanted entries

In [ ]:
valid_licenses = [
    'cc-by-3.0',
    'mozilla public license 2.0',
    'cc0 1.0 universal',
    'cc0-1.0',
    'apache-2.0',
    'bsd 3-clause',
    'mit-license',
    'odc-by-1.0',
    'apache license 2.0',
    'cc-by-3.0 unported',
    'cc-by-nc-3.0 unported',
    'cc0',
    'bsd3-clause',
    'cc-zero',
    'public domain',
    'mit license',
    'bsd 3-clause "new" or "revised" license',
    'cc0 (mostly, but can differ depending on resource)',
    'cc-by-4.0 international',
    'bsd-3-clause',
    'cc-by-nc-4.0',
    'creative commons attribution 4.0 international',
    'mit',
    'cc-by-4.0',
    'bsd-2-clause',
    'academic free license version 3.0',
    'creative commons attribution 3.0 (cc by 3.0) license',
    'cc-by-4.0',
    'bsd-3-clause',
]

In [ ]:
unclear_licenses = ['custom license', 'unlicensed', 'nan', 'none', 'unknown', 'other-open', 'unkown']

2. Load all Zenodo Record IDs from our Training Material

In [ ]:
from caching import get_zenodo_ids_from_yaml
import requests

file_url = "https://raw.githubusercontent.com/NFDI4BIOIMAGE/training/main/resources/nfdi4bioimage.yml"
yaml_file = "nfdi4bioimage.yml" 
response = requests.get(file_url)

# Download the current Training Material yaml file from the Git Repository
with open(yaml_file, "wb") as file:
    file.write(response.content)
print(f"File downloaded successfully as {yaml_file}")

# Extract the Zenodo Record IDs
zenodo_ids = get_zenodo_ids_from_yaml(yaml_file, valid_licenses, unclear_licenses)
print(f"Found {len(zenodo_ids)} Zenodo records: {zenodo_ids}")

3. Calculate and save all embeddings in a Huggingface Dataset

In [ ]:
from caching import cache_hf
from tqdm import tqdm

repo_name = "ScaDS-AI/SlideInsight_Cache_v2"
for record_id in tqdm(zenodo_ids, desc = "Zenodo Records"):
    cache_hf(record_id, token, use_api, repo_name)

4. Example on how to load the data again for a specific slide from a specific Record/Presentation

In [ ]:
from caching import load_single_hf_cache
import pandas as pd

parquet_path="hf://datasets/ScaDS-AI/SlideInsight_Cache_v2/data/train-00000-of-00001.parquet"

# function expects load_single_hf_cache(record_id, slide_number, parquet_path, pdf_number=1)  
df = load_single_hf_cache("10008464", 3, parquet_path)
df

5. How to load the whole dataset

In [ ]:
from caching import load_full_hf_cache

df_full = load_full_hf_cache(repo_name)    
df_full.head()

6. Check whether all records from the .yml file are included

In [ ]:
unique_zenodo_ids = df_full["zenodo_record_id"].unique()
print(unique_zenodo_ids)
print(f"Number of total records: {len(unique_zenodo_ids)}")

-> only 40 of all 43 records were included. Look at those that are not present in the dataset yet.

In [ ]:
missing_records = [x for x in zenodo_ids if x not in unique_zenodo_ids]

print(missing_records)

-> all of those records only have a .pptx file and no .pdf file, that's why they were not processed